# Fine-Tuning the Topic Detection Model

## === Setup ===

### Importing Libraries

In [ ]:
import sys
import pandas
import sklearn.model_selection

sys.path.append("../source")
import data_preprocessing
import transformer_model

pandas.set_option("display.max_rows", None)
pandas.set_option("display.max_columns", None)
pandas.set_option("display.max_colwidth", None)

### Loading the Dataset

In [ ]:
df = pandas.read_csv("../datasets/sustainability_goals_reannotated.csv", low_memory=False, index_col="Unnamed: 0")
df = df.dropna(subset=["Text Blocks", "Topic"])
df = df.drop_duplicates(subset=["Text Blocks"])

print("Dataset Size:", df.shape)
df.head()

## === Data Preprocessing ===

In [ ]:
df["text"] = df["Text Blocks"].copy()
df["labels"] = df["Topic"].copy()
target_values = sorted(df["Topic"].unique().tolist()) # df["Topic"].value_counts().index.tolist()
df["labels"] = df["labels"].replace(dict([(v, i) for i, v in enumerate(target_values)]))
df["labels"] = df["labels"].astype(int)

data_preprocessor = data_preprocessing.DataPreprocessing()
df = data_preprocessor.clean_text_blocks(df, "text", level="essential")
df = data_preprocessor.filter_text_blocks(df, "text", keep_only_size=(0, 300))

print("Dataset Size:", df.shape)
df.head()

## === Splitting the Dataset ===

In [ ]:
df_train, df_test = sklearn.model_selection.train_test_split(
    df,
    test_size=0.2,
    stratify=df["labels"],
    random_state=7
)

print("Train Set Size:", df_train.shape)
print(df_train["labels"].value_counts())
print("Test Set Size:", df_test.shape)
print(df_test["labels"].value_counts())

## === Training and Testing the Model ===

In [ ]:
model = transformer_model.TextClassification(target_values, name="distilroberta-base", epochs=30, learning_rate=1e-5, batch_size=16, 
                                             weight_decay=0.01, save=True, save_to="../models/topic-detection")
model.fit(df_train, df_test)

## === Inference ===

In [ ]:
model = transformer_model.TextClassification(target_values, name="distilroberta-base", load_from="../models/topic-detection")
predictions = model.predict(df_test["text"].tolist())
df_test["Predicted Topic"] = predictions["Class"].values
df_test.sample(10)